In [1]:
!pip install python-dotenv
!pip install spotipy requests python-dotenv pandas

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
from dotenv import load_dotenv

# This loads the variables from your .env file into your environment
load_dotenv()

SPOTIPY_CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
SPOTIPY_CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")

print("Keys loaded successfully!")

Keys loaded successfully!


In [4]:
import os
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import requests
import pandas as pd
import time

# Load keys safely from your .env file
load_dotenv()

# --- 1. SPOTIFY SETUP ---
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.getenv("SPOTIFY_CLIENT_ID"), 
    client_secret=os.getenv("SPOTIFY_CLIENT_SECRET")
))

# --- 2. RAPIDAPI SETUP ---
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")
RAPIDAPI_HOST = "spotify-extended-audio-features-api.p.rapidapi.com"

headers = {
    "x-rapidapi-key": RAPIDAPI_KEY,
    "x-rapidapi-host": RAPIDAPI_HOST
}

# --- 3. FAKE KWORB TEST DATA ---
# This mimics the DataFrame your Kworb scraper will eventually produce
test_data = [
    {"Song": "Blinding Lights", "Artist": "The Weeknd", "Streams": "3,500,000"},
    {"Song": "Levitating", "Artist": "Dua Lipa", "Streams": "2,100,000"}
]
kworb_df = pd.DataFrame(test_data)

final_dataset = []

# --- 4. RUN THE TEST LOOP ---
for index, row in kworb_df.iterrows():
    song_title = row['Song']
    artist_name = row['Artist']
    query = f"{song_title} {artist_name}"
    
    print(f"Searching for: {query}...")
    
    search_result = sp.search(q=query, type='track', limit=1)
    
    if search_result['tracks']['items']:
        track = search_result['tracks']['items'][0]
        track_id = track['id']
        
        url = f"https://{RAPIDAPI_HOST}/v1/audio-features/{track_id}"
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            audio_features = response.json()
            
            final_dataset.append({
                'Kworb Streams': row['Streams'],
                'Track': track['name'],
                'Artist': track['artists'][0]['name'],
                'Release Date': track['album']['release_date'],
                'Danceability': audio_features.get('danceability'),
                'Energy': audio_features.get('energy'),
                'Tempo': audio_features.get('tempo')
            })
            print(f" -> Success! Pulled audio features for {track['name']}")
        else:
            print(f" -> RapidAPI Error: {response.status_code}")
    else:
        print(f" -> Track not found on Spotify.")
        
    time.sleep(0.5)

# Convert and display results
master_df = pd.DataFrame(final_dataset)
master_df

Searching for: Blinding Lights The Weeknd...
 -> Success! Pulled audio features for Blinding Lights
Searching for: Levitating Dua Lipa...
 -> Success! Pulled audio features for Levitating


,Kworb Streams,Track,Artist,Release Date,Danceability,Energy,Tempo
0,"3,500,000",Blinding Lights,The Weeknd,2020-03-20,0.513,0.730,171.001
1,"2,100,000",Levitating,Dua Lipa,2020-03-27,0.695,0.884,103.014
